# Detect data, domain, and concept shift

**Goal.** Distinguish feature, domain, concept, and performance changes with deterministic evidence.

**Prerequisites.** Base install; optional ML extra for richer evaluation.

**Produces.** Drift metrics, a segment-mix table, performance comparisons, and a report fingerprint.

**Source size.** The default cells generate approximately 1,000 logical payments; increase duration and population together for a 10,000-payment run.

**Offline path.** All marked offline cells run without Docker or network services. Service cells are optional and explicitly marked in notebook metadata.

**Cleanup.** Outputs are written under a temporary directory; remove any local run directory if you changed the output location.


**Set up a deterministic source run**


In [ ]:
# ruff: noqa
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import polars as pl

from fraudtwin.config import load_config
from fraudtwin.generation import generate

root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs" / "minimal.yaml").exists()),
    Path.cwd(),
)
base = load_config(root / "configs" / "minimal.yaml")
# Scale the population so the bounded example produces about 1,000 payments.
population = base.population.model_copy(
    update={
        "customers": 200,
        "accounts": 300,
        "cards": 240,
        "devices": 240,
        "pix_keys": 160,
        "merchants": 60,
    }
)
simulation = base.simulation.model_copy(update={"duration_days": 10})
fraud = base.fraud.model_copy(update={"enabled": True, "target_rate": 0.05})
config = base.model_copy(
    update={"population": population, "simulation": simulation, "fraud": fraud}
)
data = generate(config, write=False)
run_id = data.run_id
payments = pl.DataFrame([item.model_dump(mode="json") for item in data.behavior.payments])
print({"run_id": run_id, "payments": len(payments), "events": len(data.behavior.payment_events)})

**Inspect schema, grain, and counts**


In [ ]:
# ruff: noqa
from fraudtwin.ml.drift import DriftConfig, compare_windows

reference = payments.to_dicts()
comparison = [
    dict(row, amount=float(row.get("amount") or 0) * 1.25, channel="shifted") for row in reference
]
drift_config = DriftConfig(
    reference_name="reference",
    comparison_name="comparison",
    minimum_samples=1,
    fields=("amount", "channel"),
)
report = compare_windows(reference, comparison, drift_config)
print(
    {
        "reference": report.reference_count,
        "comparison": report.comparison_count,
        "fingerprint": report.fingerprint,
    }
)

**Run the core operation**


In [ ]:
# ruff: noqa
metrics = pl.DataFrame([metric.model_dump(mode="json") for metric in report.metrics])
display(metrics)
print("alerts:", len(report.alerts))

**Measure and interpret the result**


In [ ]:
# ruff: noqa
segment = (
    payments.group_by([c for c in ("merchant_id", "payer_account_id") if c in payments.columns])
    .len()
    .sort("len", descending=True)
    .head(10)
)
display(segment)
print("domain shift means the population mix changed; it is not automatically model failure")

**Exercise a parameter or failure mode**


In [ ]:
# ruff: noqa
from fraudtwin.ml.drift import compare_performance

performance = compare_performance(
    {"pr_auc": 0.42, "recall_at_fpr_1pct": 0.30},
    {"pr_auc": 0.31, "recall_at_fpr_1pct": 0.20},
    config=drift_config,
    label_policy="exclude_unresolved",
)
display(pl.DataFrame([metric.model_dump(mode="json") for metric in performance]))

**Write a compact artifact and fingerprint**


In [ ]:
# ruff: noqa
with TemporaryDirectory(prefix="fraudtwin-drift-") as tmp:
    path = Path(tmp) / "drift-report.json"
    path.write_text(report.model_dump_json(indent=2), encoding="utf-8")
    print(
        {
            "report": str(path),
            "bytes": path.stat().st_size,
            "interpretation": "investigate alerts before retraining",
        }
    )

**Verify invariants and clean up**


In [ ]:
# ruff: noqa
summary = {
    "run_id": run_id,
    "payments": len(data.behavior.payments),
    "payment_events": len(data.behavior.payment_events),
    "fraud_records": len(data.behavior.fraud_records),
}
assert summary["payments"] == len(payments)
assert summary["payments"] > 0
print(json.dumps(summary, indent=2, default=str))

## Record the generated shape and tutorial contract.


In [ ]:
# ruff: noqa
active = next(
    (globals().get(name) for name in ("data", "baseline") if globals().get(name) is not None), None
)
assert active is not None
summary = {
    "tutorial_id": 11,
    "payments": len(active.behavior.payments),
    "events": len(active.behavior.payment_events),
}
print(summary)
assert summary["payments"] >= 0

## Inspect stable payment identities.


In [ ]:
# ruff: noqa
ids = [item.payment_id for item in active.behavior.payments]
assert len(ids) == len(set(ids))
print({"unique_payment_ids": len(ids)})

## Record the generated shape and tutorial contract.


In [ ]:
# ruff: noqa
active = next(
    (globals().get(name) for name in ("data", "baseline") if globals().get(name) is not None), None
)
assert active is not None
summary = {
    "tutorial_id": 11,
    "payments": len(active.behavior.payments),
    "events": len(active.behavior.payment_events),
}
print(summary)
assert summary["payments"] >= 0